In [254]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [255]:
RANDOM_STATE = 42
TARGET = "employed_status"
ID_COL = "anonymised_id"
N_SPLITS = 5

train = pd.read_csv("data/train.csv")
train = train.dropna(subset=["employed_status"])
test = pd.read_csv("data/test.csv")
groups_train = train[ID_COL]

Feature Engineering: Tenure, gated by prior employment and age x employed_lag and work_readiness_score x is_first_round

In [256]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["has_history"] = df["lag_round"].notna().astype(int)
    df["employed_lag_num"] = df["employed_lag"]  # 0/1/NaN
    df["employed_lag_x_recency"] = df["employed_lag_num"].fillna(0) * df["has_history"]

    # --- Tenure, gated by prior employment ---------------------------------
    # only treat tenure as a risk signal for rows that WERE employed last
    # round -- zero it out otherwise, so the coefficient isn't diluted by
    # unrelated non-employed zeros.
    df["tenure_lag"] = df["tenure_lag"].fillna(0)
    df["log_tenure_lag"] = np.log1p(df["tenure_lag"].clip(lower=0))
    df["log_tenure_lag_if_employed"] = df["log_tenure_lag"] * df["employed_lag_num"].fillna(0)

    df["is_first_round"] = (df["total_historical_rounds"] <= 1).astype(int)

     # --- FIX: Center age before squaring ---------------------------------
    df["age"] = df["age"].fillna(df["age"].median())

    # --- age x employed_lag --------------------------------------------
    # Age likely means something different depending on prior state: among
    # the already-employed it's closer to a tenure/seniority proxy; among
    # the not-employed it's closer to a "how long searching" proxy.
    df["age_x_employed_lag"] = df["age"] * df["employed_lag_num"].fillna(0)

    # --- work_readiness_score x is_first_round --------------------------
    # Purpose-built forward-looking score should matter most when it's the
    # ONLY forward signal available (no employed_lag / status history).
    df["work_readiness_x_first_round"] = df["work_readiness_score"].fillna(
        df["work_readiness_score"].median()
    ) * df["is_first_round"]

    return df

In [257]:
def add_seasonality_features(df: pd.DataFrame, date_col: str = "survey_date") -> pd.DataFrame:
    """Add cyclical seasonality features: sin/cos of month."""
    df = df.copy()
    
    if date_col not in df.columns:
        df["month_sin"] = 0
        df["month_cos"] = 0
        df["month_sin_x_employed_lag"] = 0
        df["month_cos_x_employed_lag"] = 0
        return df
    
    # Extract month
    month = pd.to_datetime(df[date_col]).dt.month
    
    # Cyclical encoding
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    
    # Interaction with employed_lag - season affects transitions differently
    df["month_sin_x_employed_lag"] = df["month_sin"] * df["employed_lag_num"].fillna(0)
    df["month_cos_x_employed_lag"] = df["month_cos"] * df["employed_lag_num"].fillna(0)
    
    return df

In [258]:
def make_interaction_categorical(df, col_a, col_b, new_col, min_count=None,
                                  train_ref=None, other_label="Other"):
    """Combine two categorical columns into one 'A||B' categorical.
    NaNs are stringified so 'Missing' combinations are preserved as their
    own category rather than dropped."""
    a = df[col_a].astype(str).fillna("Missing")
    b = df[col_b].astype(str).fillna("Missing")
    df[new_col] = a + "||" + b

    if min_count is not None:
        ref = train_ref if train_ref is not None else df
        counts = ref[new_col].value_counts()
        keep = set(counts[counts >= min_count].index)
        df[new_col] = df[new_col].where(df[new_col].isin(keep), other_label)
    return df

In [ ]:
def collapse_rare_categories(train_df, test_df, col, min_count=30, other_label="Other"):
    counts = train_df[col].value_counts()
    keep = set(counts[counts >= min_count].index)

    def _collapse(series):
        return series.where(series.isin(keep) | series.isna(), other_label)

    train_df[col] = _collapse(train_df[col])
    test_df[col] = _collapse(test_df[col])
    return train_df, test_df

def extract_month_from_date(df: pd.DataFrame, date_col: str = "survey_date") -> pd.Series:
    """Extract month from survey_date column."""
    if date_col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_datetime(df[date_col]).dt.month

In [261]:
train = engineer_features(train)
test = engineer_features(test)

train = add_seasonality_features(train)
test = add_seasonality_features(test)


# --- combined interaction categoricals -------------------------------
train = make_interaction_categorical(train, "gender", "status_broad_lag",
                                      "gender_x_status_lag")
test = make_interaction_categorical(test, "gender", "status_broad_lag",
                                     "gender_x_status_lag")

train = make_interaction_categorical(train, "education_level", "status_broad_lag",
                                      "education_x_status_lag")
test = make_interaction_categorical(test, "education_level", "status_broad_lag",
                                     "education_x_status_lag")

# race x education_level: sparser combo, so collapse rare cells using
# TRAIN-only counts to avoid leakage.
train = make_interaction_categorical(train, "race", "education_level",
                                      "race_x_education", min_count=50,
                                      train_ref=train)
test = make_interaction_categorical(test, "race", "education_level",
                                     "race_x_education")

# map test's raw combos through the same keep-set as train (anything not
# seen with min_count in train becomes "Other")
_keep_race_edu = set(train["race_x_education"].unique()) - {"Other"}
test["race_x_education"] = test["race_x_education"].where(
    test["race_x_education"].isin(_keep_race_edu), "Other"
)




In [ ]:
numeric_features = [
    "age", 
    "employed_lag_x_recency",
    "log_tenure_lag_if_employed",   # replaces log_tenure_lag
   "total_historical_rounds",
    "has_history",
    "is_first_round",
    "work_readiness_score",
    "work_readiness_x_first_round",     # NEW
    "age_x_employed_lag",               # NEW
    "month_sin",                     # NEW
    "month_cos",                     # NEW
    "month_sin_x_employed_lag",      # NEW - interaction
    "month_cos_x_employed_lag",      # NEW - interaction
]

categorical_features = [
    "status_broad_lag",
    "gender",
    "race",
    "province",
    "education_level",
    "gender_x_status_lag",       # NEW
    "education_x_status_lag",    # NEW
   
]

numeric_features = [c for c in numeric_features if c in train.columns]
categorical_features = [c for c in categorical_features if c in train.columns]

X_train = train[numeric_features + categorical_features]
y_train = train[TARGET].astype(int)
X_test = test[numeric_features + categorical_features]

PIPELINE

In [263]:


numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])


preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = RandomForestClassifier(
    n_estimators=600,
    max_depth=9,
    min_samples_split=60,
    min_samples_leaf=25,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)


pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", model),
])

Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups_train), 1):
    tr_df, val_df = train.iloc[tr_idx].copy(), train.iloc[val_idx].copy()

    tr_df, val_df = collapse_rare_categories(tr_df, val_df, "education_level", min_count=100)

    X_tr = tr_df[numeric_features + categorical_features]
    X_val = val_df[numeric_features + categorical_features]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    pipeline.fit(X_tr, y_tr)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(auc)
    print(f"Fold {fold}: AUC = {auc:.5f}")

print(f"\nMean CV AUC: {np.mean(fold_aucs):.5f} (+/- {np.std(fold_aucs):.5f})")

Fold 1: AUC = 0.64930
Fold 2: AUC = 0.65678
Fold 3: AUC = 0.66068
Fold 4: AUC = 0.65004
Fold 5: AUC = 0.64292

Mean CV AUC: 0.65194 (+/- 0.00619)


In [ ]:
from sklearn.metrics import roc_auc_score

# Use several trailing rounds as successive cutoffs instead of one,
# to average out the "small round" noise problem.
rounds_sorted = sorted(train["current_round"].unique())
val_rounds = rounds_sorted[-3:]  # last 3 rounds as walk-forward cutoffs

fold_aucs = []
subgroup_aucs = []  # track has_history split per fold

for cutoff in val_rounds:
    tr_df = train[train["current_round"] < cutoff].copy()
    val_df = train[train["current_round"] == cutoff].copy()

    if val_df.empty or tr_df.empty:
        continue

    # all target/frequency encodings fit on tr_df ONLY (time-safe)
    tr_df, val_df = collapse_rare_categories(tr_df, val_df, "education_level", min_count=100)

    X_tr = tr_df[numeric_features + categorical_features]
    X_val = val_df[numeric_features + categorical_features]
    y_tr = tr_df[TARGET].astype(int)
    y_val = val_df[TARGET].astype(int)

    pipeline.fit(X_tr, y_tr)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(auc)

    # subgroup check: new entrants vs returning respondents
    has_hist = val_df["has_history"].values.astype(bool)
    sub = {}
    if has_hist.sum() > 20:
        sub["returning"] = roc_auc_score(y_val[has_hist], val_probs[has_hist])
    if (~has_hist).sum() > 20:
        sub["new_entrant"] = roc_auc_score(y_val[~has_hist], val_probs[~has_hist])
    subgroup_aucs.append(sub)

    print(f"Cutoff round {cutoff}: n_val={len(val_df)}, AUC={auc:.5f}, subgroups={sub}")

print(f"\nWalk-forward mean AUC: {np.mean(fold_aucs):.5f} (+/- {np.std(fold_aucs):.5f})")

Cutoff round 6: n_val=4883, AUC=0.62330, subgroups={'returning': 0.703466796875, 'new_entrant': 0.5896144719372876}
Cutoff round 7: n_val=3199, AUC=0.64404, subgroups={'returning': 0.6499684455488501, 'new_entrant': 0.6474873845282875}
Cutoff round 8: n_val=2333, AUC=0.66364, subgroups={'returning': 0.7580363953361665, 'new_entrant': 0.6438615932883685}

Walk-forward mean AUC: 0.64366 (+/- 0.01647)


Fit the pipeline on the full training data and make predictions on the test set:

In [ ]:

train, test = collapse_rare_categories(train, test, "education_level", min_count=100)


X_train_final = train[numeric_features + categorical_features]
X_test_final = test[numeric_features + categorical_features]
pipeline.fit(X_train_final, y_train)
test_probs = pipeline.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "employed_prob": test_probs,
})

submission.to_csv("Submissions/RF_model.csv", index=False)
print("\nSaved RF_model.csv")


Saved RF_model.csv
